# prismo — differentiable PN-junction phase shifter, in the browser

This notebook runs the whole pipeline of [prismo](https://github.com/benvial/prismo) — drift-diffusion
(ChargeTransport.jl) → Soref–Bennett → optical eigenmode (gyptis/FEniCS) → MMA optimization of the doping
profile of a silicon rib-waveguide phase shifter — and shows the figures inline.

**How this differs from a local run.** The solvers normally run as two Tesseract containers
(`make run-containers`). Binder has no Docker, so here they run *in-process*: Julia and gyptis live in this
kernel's environment and `prismo` calls them directly (the `make run` / `prismo run` path). That path
authors a simpler rib mesh (no PML frame) with `prismo.waveguide_mesh`; physics, adjoint and optimizer
are identical.

**Budget.** Binder gives ~1 CPU and 1–2 GB of RAM, and stops the session after 10 minutes without
browser activity. One objective + gradient evaluation takes about a minute here, so the cells below
default to a short run (`--max-iter 20`). The headline result in the README is a 200-iteration run —
that is 2–3 h here; keep the tab open, or run it locally.

Every cell is a `prismo` CLI call, exactly what the `make` targets wrap: open a terminal from the
launcher (`File ▸ New ▸ Terminal`) to use `make run`, `prismo run --help`, or any other knob.

## 0. Environment check

In [ ]:
import os
from pathlib import Path

# The CLI writes to outputs/ relative to the working directory: run from the repository root.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("working directory:", Path.cwd())

!prismo --help | head -4
!julia --version
!python -c "import gyptis, dolfin, jax; print('gyptis', gyptis.__version__, '| jax', jax.__version__)"

In [ ]:
# Both solvers must be importable in-process; a missing one makes prismo raise when called, not stub.
from prismo.pipeline import default_components

components = default_components()
assert components.write_mesh is not None, (
    "gyptis tesseract_api did not load (FEniCS missing?)"
)
assert components.reset_chargetransport is not None, (
    "chargetransport tesseract_api did not load (julia missing?)"
)
print("gyptis + ChargeTransport components loaded in-process")

In [ ]:
import subprocess

from IPython.display import Image, display

OUT = Path("outputs")


def show(name: str, dpi: int = 110) -> None:
    """Display outputs/<name>.{png,gif,pdf} inline (PDFs rasterised with pdftoppm)."""
    for ext in ("png", "gif"):
        if (OUT / f"{name}.{ext}").exists():
            display(Image(filename=str(OUT / f"{name}.{ext}")))
            return
    pdf = OUT / f"{name}.pdf"
    if not pdf.exists():
        print(f"{pdf} not found (yet)")
        return
    stem = OUT / f"{name}_render"
    subprocess.run(
        ["pdftoppm", "-png", "-r", str(dpi), "-singlefile", str(pdf), str(stem)],
        check=True,
    )
    display(Image(filename=f"{stem}.png"))

## 1. Gradient check

The composed adjoint (discrete adjoint through the Julia drift-diffusion solve, Hellmann–Feynman
eigen-adjoint through the FEniCS eigensolve, JAX in between) against central finite differences along
a random direction in design space. The first call pays the Julia JIT and the FEniCS form compilation
(~1 min); a full 3-direction × 12-step check is `prismo validate-gradient` with no arguments.

In [ ]:
!prismo validate-gradient --n-directions 1 --n-steps 4

In [ ]:
show("gradient_validation")

## 2. Optimize the doping profile

MMA on the signed design field at every silicon mesh node, density-filtered with radius `--r-min`,
maximizing Δn_eff between 0 V and −5 V (`--loss-weight` adds the modal free-carrier loss to the
objective). Progress prints per iteration; `outputs/checkpoint.json` is written after each one, so an
interrupted run still leaves its best design and history behind.

In [ ]:
!prismo run --max-iter 20 --seed lateral

## 3. Results

In [ ]:
import json

history = json.loads((OUT / "checkpoint.json").read_text())["history"]
best = max(history, key=lambda e: e.get("objective", e["delta_n_eff"]))
print(f"{len(history)} evaluations; best Δn_eff = {best['delta_n_eff']:.3e}")
if "modal_loss_db_cm" in best:
    print(f"  modal loss = {best['modal_loss_db_cm']:.4g} dB/cm")

In [ ]:
show("convergence")

In [ ]:
show("doping_field")

In [ ]:
show("mode_field")
show("depletion_field")

In [ ]:
# Net doping at every evaluation, one frame per history record (rebuilt by `prismo animate`).
show("doping_evolution")

## 4. Knobs

Same CLI as the README; uncomment one and run. `prismo run --help` lists everything.

In [ ]:
# Trade Δn_eff against modal loss (adds the loss_convergence and tradeoff figures):
# !prismo run --max-iter 20 --loss-weight 4e-6
# Start from a U junction, contacts 0.5 µm from the rib:
# !prismo run --max-iter 20 --seed u --contact-offset 0.5
# Optimize the first higher-order mode:
# !prismo run --max-iter 20 --mode-index 1
# Re-render doping_evolution.{gif,mp4} from outputs/checkpoint.json:
# !prismo animate --fps 4
# Objective smoothness line scan around the optimum:
# !prismo probe-objective --design outputs/checkpoint.json

In [ ]:
# show("loss_convergence"); show("tradeoff")